# Session 2: calibrating a sensor with a line
**A. Real-world problem.** A cheap temperature sensor does not report the true temperature:
it reports a number that is roughly a straight-line function of it, plus noise. To trust it
you *calibrate* it: compare its readings with trusted reference temperatures, fit a line,
and then use the line backwards to turn a new reading into a temperature with an
uncertainty. This is general education about measurement, not a procedure for a
safety-critical instrument.

**Plan.** (1) simulate a sensor with known gain, offset and noise, (2) fit the line and its
standard errors, (3) invert it, (4) separate repeatability from reproducibility, (5) add
quantisation, (6) bias versus random error, (7) how many repeats do we need?

In [ ]:
import math
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

# the made-up sensor: reading = GAIN * true + OFFSET + noise
GAIN = 1.02        # reading per degree C (no unit: degrees of reading per degree of truth)
OFFSET = -0.80     # reading at a true temperature of 0 degrees C (degrees C)
NOISE_SD = 0.30    # random scatter of one reading (degrees C)
REF = np.arange(0.0, 101.0, 10.0)   # 11 reference temperatures, 0 to 100 degrees C

## Step 1: simulate the sensor

In [ ]:
def sensor(true, rng, gain=GAIN, offset=OFFSET, sd=NOISE_SD):
    """Reading(s) of the made-up sensor at true temperature(s)."""
    true = np.asarray(true, dtype=float)
    return gain * true + offset + rng.normal(0.0, sd, true.shape)

## Step 2: fit the calibration line and its standard errors
With x = reference temperatures (n points) and y = readings, least squares gives

    slope m = Sxy / Sxx,   intercept c = ybar - m * xbar,
    residual sd s = sqrt( sum(residual^2) / (n - 2) ),
    SE(m) = s / sqrt(Sxx),   SE(c) = s * sqrt(1/n + xbar^2 / Sxx),

where Sxx = sum (x - xbar)^2 and Sxy = sum (x - xbar)(y - ybar). `np.polyfit(x, y, 1)`
gives the same m and c; the standard errors we add ourselves.

In [ ]:
def fit_calibration(x, y):
    """Returns dict with slope, intercept, their standard errors, residual sd s, n, xbar, Sxx."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    xbar, ybar = x.mean(), y.mean()
    sxx = float(np.sum((x - xbar) ** 2))
    sxy = float(np.sum((x - xbar) * (y - ybar)))
    m = sxy / sxx
    c = ybar - m * xbar
    resid = y - (m * x + c)
    s = math.sqrt(float(np.sum(resid ** 2)) / (n - 2))
    return {"slope": m, "intercept": c, "se_slope": s / math.sqrt(sxx),
            "se_intercept": s * math.sqrt(1.0 / n + xbar ** 2 / sxx),
            "s": s, "n": n, "xbar": xbar, "sxx": sxx}


def bootstrap_se(x, y, reps, seed):
    """Residual bootstrap: refit after resampling the residuals. Returns (sd of slopes, sd of intercepts)."""
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    m, c = np.polyfit(x, y, 1)
    resid = np.asarray(y) - (m * x + c)
    ms, cs = [], []
    for _ in range(reps):
        yb = m * x + c + rng.choice(resid, size=len(x), replace=True)
        mb, cb = np.polyfit(x, yb, 1)
        ms.append(mb)
        cs.append(cb)
    return float(np.std(ms, ddof=1)), float(np.std(cs, ddof=1))

## Step 3: invert the line (inverse prediction)
The sensor shows a reading y0 (the mean of k fresh readings). The best estimate of the
true temperature is x0 = (y0 - c) / m. Its uncertainty has three parts: the noise of the
new reading(s), the uncertainty of the line's height, and of its tilt:

    u(x0) = (s / |m|) * sqrt( 1/k + 1/n + (x0 - xbar)^2 / Sxx )

In [ ]:
def invert(fit, y0, k=1):
    """Estimated true temperature and its standard uncertainty for a mean reading y0 of k readings."""
    x0 = (y0 - fit["intercept"]) / fit["slope"]
    u = fit["s"] / abs(fit["slope"]) * math.sqrt(1.0 / k + 1.0 / fit["n"] + (x0 - fit["xbar"]) ** 2 / fit["sxx"])
    return x0, u

## Step 4: repeatability versus reproducibility
*Repeatability*: the scatter when you repeat the measurement at once, same person, same
day. *Reproducibility*: the scatter when conditions change (another day, another set-up).
Suppose each day the sensor offset shifts by a random amount with sd DAY_SD. Then
reproducibility sd = sqrt(repeatability^2 + DAY_SD^2).

In [ ]:
DAY_SD = 0.50   # day-to-day shift of the offset (degrees C)


def day_data(days, per_day, seed, sd=NOISE_SD, day_sd=DAY_SD):
    """Readings of a constant truth: array (days, per_day) = daily shift + noise."""
    rng = np.random.default_rng(seed)
    return rng.normal(0.0, day_sd, (days, 1)) + rng.normal(0.0, sd, (days, per_day))


def repeat_and_reproduce(data):
    """(repeatability sd = pooled within-day sd, reproducibility sd = sd of all readings)."""
    within = float(np.sqrt(np.mean(data.var(axis=1, ddof=1))))
    total = float(data.std(ddof=1))
    return within, total

## Step 5: resolution and quantisation
A digital sensor turns a voltage into one of 2^N numbers (N bits). Over a range from lo to
hi the step between neighbouring numbers is LSB = (hi - lo) / 2^N ("least significant bit").
Rounding to the nearest step makes an error between -LSB/2 and +LSB/2, with standard
deviation LSB / sqrt(12) when the signal moves across many steps.

In [ ]:
def lsb(lo, hi, bits):
    return (hi - lo) / 2 ** bits


def quantise(v, lo, hi, bits):
    """Round values to the centre of their ADC step; values outside lo..hi are clipped."""
    q = lsb(lo, hi, bits)
    code = np.clip(np.floor((np.asarray(v, dtype=float) - lo) / q), 0, 2 ** bits - 1)
    return lo + (code + 0.5) * q


def quantisation_rms(lo, hi, bits, n, seed):
    """Measured rms rounding error for n uniformly random values in the range."""
    rng = np.random.default_rng(seed)
    v = rng.uniform(lo, hi, n)
    return float(np.sqrt(np.mean((quantise(v, lo, hi, bits) - v) ** 2)))


def mean_of_quantised(true, noise_sd, reps, lo, hi, bits, seed):
    """Mean of many quantised readings of a constant truth with Gaussian noise before the ADC."""
    rng = np.random.default_rng(seed)
    return float(np.mean(quantise(true + rng.normal(0.0, noise_sd, reps), lo, hi, bits)))

## Step 6: bias versus random error
Random error shrinks like 1 / sqrt(N) when you average N readings. A bias (a constant
error such as a wrong offset) does not shrink at all.

In [ ]:
def average_error_stats(bias, sd, N, trials, seed):
    """(mean, sd) of the error of the N-reading average over many trials."""
    rng = np.random.default_rng(seed)
    err = bias + rng.normal(0.0, sd, (trials, N)).mean(axis=1)
    return float(err.mean()), float(err.std(ddof=1))

## Step 7: design - how many repeats?
To get the 95 percent half-width of the mean down to h (degrees C) when one reading has
known scatter sd, we need 1.96 * sd / sqrt(N) <= h, so N >= (1.96 * sd / h)^2. We round up.

In [ ]:
Z95 = 1.959964   # the two-sided 95 percent point of the standard normal


def repeats_needed(sd, half_width, z=Z95):
    return int(math.ceil((z * sd / half_width) ** 2 - 1e-12))


def coverage(sd, N, half_width):
    """Chance that the mean of N readings lies within +-half_width of the truth (random error only)."""
    zz = half_width * math.sqrt(N) / sd
    return math.erf(zz / math.sqrt(2.0))

## Step 8: see it
Left: the calibration data, the fitted line and the true line. Right: the staircase of a
6-bit ADC. Second figure: averaging helps random error, not bias.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    rng = np.random.default_rng(2)
    y = sensor(REF, rng)
    f = fit_calibration(REF, y)
    print("fit: slope %.4f +- %.4f, intercept %.3f +- %.3f, s = %.3f" % (
        f["slope"], f["se_slope"], f["intercept"], f["se_intercept"], f["s"]))
    print("polyfit:", np.polyfit(REF, y, 1).round(4))
    print("bootstrap se:", np.round(bootstrap_se(REF, y, 2000, 5), 4))
    x0, u = invert(f, 40.0, k=1)
    print("reading 40.0 -> true %.2f +- %.2f degrees C" % (x0, u))
    print("repeats needed for 0.1 degrees:", repeats_needed(NOISE_SD, 0.1))
    print("LSB for 0..100 at 8 bits: %.4f, sd %.4f" % (lsb(0, 100, 8), lsb(0, 100, 8) / math.sqrt(12)))

    fig, axs = plt.subplots(1, 2, figsize=(10, 4))
    ax = axs[0]
    ax.scatter(REF, y, s=18, label="readings")
    xs = np.linspace(0, 100, 50)
    ax.plot(xs, f["slope"] * xs + f["intercept"], label="fitted line")
    ax.plot(xs, GAIN * xs + OFFSET, "--", color="gray", label="true line (we do not know it)")
    ax.set_xlabel("reference temperature (degrees C)")
    ax.set_ylabel("sensor reading (degrees C)")
    ax.set_title("Calibration line")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax = axs[1]
    t = np.linspace(20, 30, 400)
    ax.plot(t, t, color="gray", lw=1, label="ideal")
    ax.step(t, quantise(t, 0, 100, 6), where="mid", label="6-bit ADC (LSB %.3f)" % lsb(0, 100, 6))
    ax.set_xlabel("true value (degrees C)")
    ax.set_ylabel("digital reading (degrees C)")
    ax.set_title("Quantisation staircase")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "calibration_and_adc.png", dpi=120)

    Ns = np.array([1, 4, 16, 64, 256])
    sds, means = [], []
    for N in Ns:
        mm, ss = average_error_stats(0.5, NOISE_SD, int(N), 4000, 8)
        means.append(mm)
        sds.append(ss)
        print("N = %3d: mean error %.3f, scatter of the average %.3f (theory %.3f)" % (N, mm, ss, NOISE_SD / math.sqrt(N)))
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.loglog(Ns, sds, "o-", label="random error (scatter of the average)")
    ax2.loglog(Ns, np.abs(means), "s-", label="bias (mean error)")
    ax2.set_xlabel("readings averaged, N")
    ax2.set_ylabel("error (degrees C)")
    ax2.set_title("Averaging beats random error, not bias")
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3, which="both")
    fig2.tight_layout()
    fig2.savefig(OUT / "average_vs_bias.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change `REF` in `lesson.py` to only 4 points (0, 10, 90, 100) and compare `se_slope` with the 11-point value 0.00286. Predict first, then run.
2. **Stretch.** Use `invert(fit, y0, k)` for readings 0, 50 and 100 and print the uncertainties. Explain the pattern.
3. **Explain.** Explain why `mean_of_quantised` with noise 0.0 keeps an error of about 0.13 degrees C at a true value of 20.25, but with noise 0.4 the average is within 0.01.

## Assessment
- **A. Conceptual.** Why is the uncertainty of the inverted temperature smallest in the middle of the calibrated range and larger near the ends? Why can you not reduce it to zero by averaging the new readings?
- **B. Numerical.** A sensor has scatter sd = 1.0 degrees C. How many readings must be averaged to know the mean to +-0.2 degrees C at 95 percent, and how many for +-0.1?
- **C. Graph and simulation interpretation.** In the average-versus-bias figure the scatter line falls with slope -1/2 on log axes while the bias line is flat. What does each tell you, and which could you reduce by taking more readings?
- **D. Design and optimisation challenge.** Design: a 10-bit ADC over 0 to 100 degrees C is available. Its sensor noise sd is 0.30 degrees C. Is the quantisation noise a problem compared with the sensor noise? Would an 8-bit ADC be acceptable? Justify with numbers.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    xb = float(L.REF.mean())
    sxx = float(np.sum((L.REF - xb) ** 2))
    checks.append(("reference grid: n = 11, xbar = 50, Sxx = 11000", len(L.REF) == 11 and xb == 50.0 and sxx == 11000.0))

    # fit agrees with np.polyfit on one noisy dataset
    rng = np.random.default_rng(2)
    y = L.sensor(L.REF, rng)
    f = L.fit_calibration(L.REF, y)
    pf = np.polyfit(L.REF, y, 1)
    checks.append(("fit_calibration slope and intercept equal np.polyfit",
                   abs(f["slope"] - pf[0]) < 1e-9 and abs(f["intercept"] - pf[1]) < 1e-9))
    # exact line: zero residual
    fx = L.fit_calibration(L.REF, 2.0 * L.REF + 3.0)
    checks.append(("edge case: noise-free data gives the exact line and s = 0",
                   abs(fx["slope"] - 2.0) < 1e-9 and abs(fx["intercept"] - 3.0) < 1e-9 and fx["s"] < 1e-9))

    # Monte Carlo of the whole calibration experiment, 20000 repeats, vectorised
    rng = np.random.default_rng(1)
    R = 20000
    Y = L.GAIN * L.REF + L.OFFSET + rng.normal(0, L.NOISE_SD, (R, 11))
    m = ((L.REF - xb) * (Y - Y.mean(1, keepdims=True))).sum(1) / sxx
    c = Y.mean(1) - m * xb
    se_m = L.NOISE_SD / math.sqrt(sxx)
    se_c = L.NOISE_SD * math.sqrt(1 / 11 + xb ** 2 / sxx)
    checks.append(("hand calculation: SE(slope) = 0.3 / sqrt(11000) = 0.00286", abs(se_m - 0.00286) < 0.00001))
    checks.append(("simulated scatter of the slope matches the formula (within 3 percent)",
                   abs(m.std(ddof=1) / se_m - 1) < 0.03))
    checks.append(("simulated scatter of the intercept (0.169) matches the formula (within 3 percent)",
                   abs(c.std(ddof=1) / se_c - 1) < 0.03 and abs(se_c - 0.169) < 0.001))
    checks.append(("the fitted slope and intercept are unbiased: means 1.020 and -0.800",
                   abs(m.mean() - L.GAIN) < 0.0001 and abs(c.mean() - L.OFFSET) < 0.005))

    # bootstrap check on the single dataset
    bm, bc = L.bootstrap_se(L.REF, y, 2000, 5)
    checks.append(("bootstrap standard errors agree with the formulas within 15 percent",
                   abs(bm / f["se_slope"] - 1) < 0.15 and abs(bc / f["se_intercept"] - 1) < 0.15))

    # worked example: inverse prediction with the nominal line
    nominal = {"slope": 1.02, "intercept": -0.8, "s": 0.3, "n": 11, "xbar": 50.0, "sxx": 11000.0}
    x0, u = L.invert(nominal, 30.4, 1)
    checks.append(("worked example: reading 30.4 -> 30.59 degrees C, uncertainty 0.31",
                   abs(x0 - 30.588) < 0.001 and abs(u - 0.312) < 0.001))
    x4, u4 = L.invert(nominal, 30.4, 4)
    checks.append(("averaging k = 4 new readings cuts the uncertainty to 0.18", abs(u4 - 0.180) < 0.001))
    # simulated inverse prediction at true 30 degrees C
    ynew = L.GAIN * 30.0 + L.OFFSET + rng.normal(0, L.NOISE_SD, R)
    xh = (ynew - c) / m
    th = L.NOISE_SD / L.GAIN * math.sqrt(1 + 1 / 11 + (30.0 - xb) ** 2 / sxx)
    checks.append(("simulated inverse-prediction scatter at 30 degrees C matches the formula (within 3 percent)",
                   abs(xh.std(ddof=1) / th - 1) < 0.03))
    checks.append(("uncertainty is smallest at the middle of the range (50) and larger at 0",
                   L.invert(nominal, 1.02 * 50 - 0.8)[1] < L.invert(nominal, -0.8)[1]))

    # repeatability / reproducibility
    rep, repro = L.repeat_and_reproduce(L.day_data(2000, 10, 3))
    checks.append(("repeatability is about 0.30 degrees C", abs(rep - 0.30) < 0.01))
    checks.append(("reproducibility is about sqrt(0.30^2 + 0.50^2) = 0.58 degrees C",
                   abs(repro - math.sqrt(0.09 + 0.25)) < 0.02 and repro > rep))

    # quantisation
    checks.append(("8 bits over 0..100: LSB = 0.390625 degrees C", abs(L.lsb(0, 100, 8) - 0.390625) < 1e-12))
    checks.append(("8 bits: LSB / sqrt(12) = 0.113 degrees C", abs(L.lsb(0, 100, 8) / math.sqrt(12) - 0.1128) < 0.0001))
    ok = True
    for bits in (6, 8, 10, 12):
        pred = L.lsb(0, 100, bits) / math.sqrt(12)
        ok &= abs(L.quantisation_rms(0, 100, bits, 200000, 4) / pred - 1) < 0.01
    checks.append(("measured rounding error equals LSB / sqrt(12) within 1 percent for 6, 8, 10, 12 bits", ok))
    checks.append(("each extra bit halves the step: 10 bits has LSB 0.0977", abs(L.lsb(0, 100, 10) - 0.09765625) < 1e-12))
    q = L.quantise(np.array([-5.0, 105.0]), 0, 100, 8)
    checks.append(("edge case: values outside the range are clipped to the first and last step",
                   abs(q[0] - 0.5 * 0.390625) < 1e-12 and abs(q[1] - (100 - 0.5 * 0.390625)) < 1e-12))
    quiet = L.mean_of_quantised(20.25, 0.0, 1000, 0, 100, 8, 1)
    noisy = L.mean_of_quantised(20.25, 0.4, 200000, 0, 100, 8, 1)
    checks.append(("no noise: averaging quantised readings of 20.25 keeps an error of 0.13 (no help from averaging)",
                   abs((quiet - 20.25) - (-0.1328)) < 0.001))
    checks.append(("noise about one step wide (0.4) lets averaging remove the rounding error (error below 0.01)",
                   abs(noisy - 20.25) < 0.01))

    # bias vs random
    ok_b, ok_r = True, True
    for N in (1, 4, 16, 64, 256):
        mm, ss = L.average_error_stats(0.5, L.NOISE_SD, N, 4000, 8)
        ok_b &= abs(mm - 0.5) < 0.01
        ok_r &= abs(ss / (L.NOISE_SD / math.sqrt(N)) - 1) < 0.06
    checks.append(("averaging N = 1..256 readings leaves the bias at 0.50 (within 0.01)", ok_b))
    checks.append(("scatter of the average follows 0.30 / sqrt(N) (within 6 percent)", ok_r))

    # design
    checks.append(("worked example: 0.1 degrees at 95 percent with sd 0.30 needs N = 35 readings",
                   L.repeats_needed(0.30, 0.1) == 35))
    checks.append(("hand calculation: (1.96 * 0.30 / 0.1)^2 = 34.57, rounded up to 35",
                   abs((1.96 * 0.30 / 0.1) ** 2 - 34.57) < 0.01))
    checks.append(("N = 35 reaches 95 percent (0.9514), N = 34 does not (0.9481)",
                   L.coverage(0.30, 35, 0.1) >= 0.95 > L.coverage(0.30, 34, 0.1)
                   and abs(L.coverage(0.30, 35, 0.1) - 0.9514) < 0.0001))
    rng = np.random.default_rng(9)
    hit = np.mean(np.abs(rng.normal(0, 0.30, (40000, 35)).mean(1)) <= 0.1)
    checks.append(("simulation: the mean of 35 readings is within 0.1 of the truth about 95 percent of the time",
                   abs(hit - 0.9514) < 0.005))
    checks.append(("halving the target half-width needs 4 times the readings (sd 1: 385 for 0.1, 97 for 0.2)",
                   L.repeats_needed(1.0, 0.1) == 385 and L.repeats_needed(1.0, 0.2) == 97))
    checks.append(("edge case: a loose target needs only 1 reading (sd 0.3, half-width 0.9 -> 1)",
                   L.repeats_needed(0.3, 0.9) == 1))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
